%md
#Bronze Layer — Raw Data Ingestion
### Notebook: 01_bronze_ingestion
**Pipeline:** Bank Customer Churn Analytics  
**Author:** Harshkumar Patel 
**Date:** 29/06/2026

---

## What this notebook does
This is the first layer of our Medallion Architecture. The Bronze layer is responsible for:
- Reading the raw CSV file from our Databricks Volume (landing zone)
- Adding metadata columns to track when and where the data came from
- Writing the data as a Delta table — with zero transformations
- Running basic data quality checks to confirm the data landed correctly

## Bronze Layer Rule
> Land the data exactly as it came. No cleaning. No fixing. No transforming.
> The Bronze table is a digital photocopy of the source file.

## Source
- **File:** bank_churn_dataset.csv  
- **Location:** /Volumes/workspace/default/churn_data/bank_churn_dataset.csv  
- **Rows:** 80,000  
- **Columns:** 26  

## Target
- **Database:** churn_bronze  
- **Table:** churn_bronze.raw_bank_customers  
- **Format:** Delta Lake  
- **Partitioned by:** origin_province

In [0]:
# creating 3 databases to represent the medallion architecture
# bronze = raw data, silver = cleaned data, gold = business KPIs
# keeps everything organised and separated by purpose

spark.sql("CREATE DATABASE IF NOT EXISTS churn_bronze")
spark.sql("CREATE DATABASE IF NOT EXISTS churn_silver")
spark.sql("CREATE DATABASE IF NOT EXISTS churn_gold")

# Verify all 3 were created successfully
spark.sql("SHOW DATABASES").show()

## Step 1: Define Source File Path
Store the file path in a variable so we only need to change it in one place if the file ever moves.

In [0]:
# storing the file path in a variable so i dont have to 
# repeat the long path again and again in the notebook

FILE_PATH = "/Volumes/workspace/default/churn_data/bank_churn_dataset.csv"

%md
## Step 2: Read Raw CSV into a Spark DataFrame
We read the file using Spark — this does NOT load it into memory like pandas.
Spark reads it in a distributed way across the cluster.

In [0]:


df_raw = (spark.read                                # reading the csv file into spark
    .option("header", "true")
    .option("inferSchema", "true")                  # inferSchema automatically detects and assigns the correct data type for each column
    .csv(FILE_PATH))

# Quick confirmation of what we read
print(f"Rows: {df_raw.count()}")
print(f"Columns: {len(df_raw.columns)}")

# Preview the first 5 rows
df_raw.display()

%md
## Step 3: Inspect the Schema
printSchema() shows us what data type Spark assigned to each column.
This is important — we need to know if dates were read as dates,
numbers as numbers, and flags as booleans before we write to Delta.

In [0]:
# checking the schema to make sure spark assigned 
# the correct data type to each column
# for example exit should be boolean, balance should 
# be integer and last_active_date should be date


df_raw.printSchema()

%md
## Step 4: Add Metadata Columns and Write Bronze Delta Table
We add 3 metadata columns before writing — these track the data's
journey through the pipeline. This is called **data lineage**.
Then we write the result as a Delta Lake table.

In [0]:
# adding 3 extra columns to keep a log of when, where 
# and what layer the data came from - this is called data lineage

# saving as delta instead of csv because delta has 
# better features like ACID transactions and query performance

from pyspark.sql.functions import current_timestamp, lit, col
# Add metadata columns to the raw DataFrame
df_bronze = (df_raw
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_pipeline_layer", lit("bronze")))
# Write to Delta Lake as a managed table in churn_bronze database
(df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("origin_province")                                 #partitionBy to partiton based on diffrent provinces
    .saveAsTable("churn_bronze.raw_bank_customers"))

print(f"Bronze table written successfully")
print(f"Total rows: {df_bronze.count():,}")

%md
## Step 5: Data Quality Checks
We verify the Bronze table was written correctly.
A good data engineer always validates their output — never assume it worked.
We check:
- Row count matches the source
- No nulls in critical columns
- Churn split looks realistic

In [0]:

# Confirm row count
count = spark.table("churn_bronze.raw_bank_customers").count()
print(f"Rows in Delta table: {count:,}")

# Null check on critical columns
spark.sql("""
    SELECT
        SUM(CASE WHEN id IS NULL THEN 1 ELSE 0 END)   AS null_id,
        SUM(CASE WHEN exit IS NULL THEN 1 ELSE 0 END)  AS null_churn,
        SUM(CASE WHEN age IS NULL THEN 1 ELSE 0 END)   AS null_age,
        SUM(CASE WHEN balance IS NULL THEN 1 ELSE 0 END) AS null_balance
    FROM churn_bronze.raw_bank_customers
""").display()

spark.sql("""
    SELECT 
        exit AS churned,
        COUNT(*) AS customer_count
    FROM churn_bronze.raw_bank_customers
    GROUP BY exit
""").display()

%md
##  Bronze Layer Complete

| Check | Result |
|-------|--------|
| Rows ingested | 80,000 |
| Nulls in critical columns | 0 |
| Churn rate | 18% (14,400 churned) |
| Target table | churn_bronze.raw_bank_customers |
| Storage format | Delta Lake |
| Partitioned by | origin_province |
| Metadata columns added | 3 |

**Next step:** Silver Layer — `02_silver_transformation`  
Clean the data, rename columns, engineer new features, and build the star schema.